In [ ]:
import subprocess, sys, os, time, threading
T_NB = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh(f"{sys.executable} -m pip install -q 'transformers==5.20.2' 'peft==0.21.2' 'trl==0.29.1' accelerate bitsandbytes pycocoevalcap 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1")
sh(f'{sys.executable} -c "import trl, transformers, peft, torch; print(trl.__version__, transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0)); assert (trl.__version__, transformers.__version__, peft.__version__) == (\'0.29.1\', \'5.20.2\', \'0.21.2\'), \'lech phien ban\'; assert torch.cuda.device_count() == 2, \'can 2 GPU\'"')


In [ ]:
import glob, json, shutil, hashlib
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
RUNS = ['scst_spice_101', 'grpo_cider_101']
SRC = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/gen_test_grpo.py", recursive=True)
       if all(os.path.exists(os.path.join(os.path.dirname(p), f)) for f in ("grpo_spice.py", "build_branch_data.py", "preds_s1_seed101.jsonl"))]
assert len(SRC) == 1, f"DUNG: can dung mot grpo-spice-script, thay {SRC}"
SRC = SRC[0]
for f in ("grpo_spice.py", "build_branch_data.py", "gen_test_grpo.py"):
    shutil.copy(f"{SRC}/{f}", f"{W}/{f}")
assert md5(f"{W}/grpo_spice.py") == "07ea87b6d156d1faa391a4274dffd7cf", "DUNG: grpo_spice.py ban cu"
assert md5(f"{W}/gen_test_grpo.py") == "b18ec1aa3fd037ddffb9411db63c2a3f", "DUNG: gen_test_grpo.py lech may nha"
S1P = f"{SRC}/preds_s1_seed101.jsonl"
assert sum(1 for _ in open(S1P)) == 6958
KIT = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/tai_dung_cham.py", recursive=True)]
assert len(KIT) == 1, f"DUNG: can dung mot luoi-kit, thay {KIT}"
KIT = KIT[0]
shutil.copy(f"{KIT}/tai_dung_cham.py", W)
NGUON = sorted(glob.glob(f"{KIT}/raw_*.jsonl"))
assert len(NGUON) == 4, NGUON
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d)
MERGED = f"{W}/s1_merged"
CK = {}
for r in RUNS:
    k = [os.path.dirname(f) for f in glob.glob(f"/kaggle/input/**/{r}/checkpoint-500/adapter_model.safetensors", recursive=True)]
    assert len(k) == 1, f"DUNG: can dung mot checkpoint-500 cua {r}, thay {k}"
    assert json.load(open(f"{k[0]}/trainer_state.json"))["global_step"] == 500, f"DUNG: {r} khong phai buoc 500"
    CK[r] = k[0]
    print(r, "md5 adapter", md5(f"{k[0]}/adapter_model.safetensors"), flush=True)
    for p in glob.glob(f"/kaggle/input/**/pred_{r}_test.jsonl", recursive=True) + glob.glob(f"/kaggle/input/**/score_{r}_test_raw.jsonl*", recursive=True):
        shutil.copyfile(p, os.path.join(W, os.path.basename(p))); print("chep tiep", os.path.basename(p), flush=True)
best = None
for t in glob.glob("/kaggle/input/**/test_ac/test.jsonl", recursive=True):
    root = os.path.dirname(t)
    if sum(1 for _ in open(t)) == 6958 and len(glob.glob(f"{root}/images/*.png")) >= 4463 and os.path.exists(f"{root}/ocr.jsonl"):
        best = root
assert best, "DUNG: khong thay test_ac du anh"
TA = best
assert md5(f"{TA}/test.jsonl") == "da58299ee551a926a21cbecb5232bf78", "DUNG: test.jsonl lech may nha"
PKG = os.path.dirname(os.path.dirname(os.path.dirname(TA)))
shutil.copytree(f"{PKG}/harness", f"{W}/thesis/harness", dirs_exist_ok=True, ignore=shutil.ignore_patterns("images"))
link = f"{W}/thesis/harness/dg1_cache/test_ac/images"
if os.path.lexists(link): os.remove(link)
os.symlink(f"{TA}/images", link)
WS = f"{W}/thesis"
print("SRC", SRC, "· BUNDLE", BUNDLE, "· TA", TA, flush=True)


In [ ]:
CAT = 3600 * 5.0
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1", "PYTORCH_ALLOC_CONF": "expandable_segments:True"}
HET = threading.Event()
def chay(cmd, log, cwd, gpu):
    t0 = time.time()
    with open(log, "a") as f:
        q = subprocess.Popen(cmd, cwd=cwd, stdout=f, stderr=subprocess.STDOUT, env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)})
        while q.poll() is None:
            time.sleep(120)
            L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
            print(f"{(time.time()-T_NB)/3600:5.2f} h · gpu{gpu} · {os.path.basename(log)} · {L[-1][:110] if L else 'chua co dong'}", flush=True)
            if time.time() - T_NB > CAT:
                print("⛔ het gio, dung de Output kip luu", flush=True); HET.set(); q.terminate(); q.wait(); return None
    print(f"--- {os.path.basename(log)} · ma thoat {q.returncode} · {(time.time()-t0)/3600:.2f} h", flush=True)
    return q.returncode
r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", MERGED], cwd=W, capture_output=True, text=True)
print((r.stdout + r.stderr)[-800:], flush=True); assert r.returncode == 0, "DUNG: hoa loi"
TA_ARGS = ["--recs", f"{TA}/test.jsonl", "--ocr", f"{TA}/ocr.jsonl", "--images", f"{TA}/images", "--tap-only", "--no-q4"]
if os.path.exists(f"{W}/kiem_hoa_test.jsonl"): os.remove(f"{W}/kiem_hoa_test.jsonl")
chay(["python", "gen_test_grpo.py", "--bundle", BUNDLE, "--merged", MERGED, "--s1", S1P, "--n", "20", "--out", f"{W}/kiem_hoa_test.jsonl"] + TA_ARGS, f"{W}/kiem_hoa_test.log", W, 0)
dong = [l for l in open(f"{W}/kiem_hoa_test.log") if "[kiểm hoà]" in l]
print(dong, flush=True)
assert dong and int(dong[-1].split("]")[1].split("/")[0]) >= 16, "DUNG: kiem hoa duoi 16/20"
KQ = {}
def nhanh(r, gpu):
    try:
        PRED, SC = f"{W}/pred_{r}_test.jsonl", f"{W}/score_{r}_test.json"
        RAW = SC.replace(".json", "_raw.jsonl")
        if chay(["python", "gen_test_grpo.py", "--bundle", BUNDLE, "--merged", MERGED, "--ckpt", CK[r], "--out", PRED] + TA_ARGS, f"{W}/gen_{r}.log", W, gpu) != 0:
            KQ[r] = "gen dung"; return
        n = sum(1 for _ in open(PRED)); assert n == 4463, f"{r}: {n} cau"
        rr = subprocess.run(["python", "tai_dung_cham.py", "gieo", "--preds", PRED, "--raw", RAW, "--nguon"] + NGUON, cwd=W, capture_output=True, text=True)
        print(r, rr.stdout[-400:], rr.stderr[-400:], flush=True); assert rr.returncode == 0, "gieo loi"
        if chay(["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground", "--preds", PRED, "--out", SC], f"{W}/cham_{r}.log", WS, gpu) != 0:
            KQ[r] = "cham dung"; return
        rr = subprocess.run(["python", "tai_dung_cham.py", "kiem", "--raw", RAW, "--nguon"] + NGUON, cwd=W, capture_output=True, text=True)
        print(r, rr.stdout[-400:], rr.stderr[-400:], flush=True); assert rr.returncode == 0, "bo cham khong tai lap"
        R = [json.loads(l) for l in open(RAW)]
        KQ[r] = f"{len(R)} buoc · exec {sum(int(x.get('executable', 0)) for x in R)}"
    except Exception as e:
        KQ[r] = f"LOI {e}"; print("⚠️", r, e, flush=True)
TH = [threading.Thread(target=nhanh, args=(r, g)) for g, r in enumerate(RUNS)]
for t in TH: t.start(); time.sleep(60)
for t in TH: t.join()
print("===== KET QUA", KQ, flush=True)
shutil.rmtree(MERGED, ignore_errors=True); shutil.rmtree(f"{W}/thesis", ignore_errors=True)
import zipfile
fs = [f for f in glob.glob(f"{W}/*") if os.path.isfile(f) and not f.endswith(".zip")]
with zipfile.ZipFile(f"{W}/luoi_test_u_1.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in fs: z.write(f, os.path.basename(f))
print("da zip", len(fs), "tep", flush=True)
assert all("exec" in v or "dung" in v for v in KQ.values()) and len(KQ) == 2, f"co luot loi {KQ}"
